# Python 语言核心 — 高级工程师面试精讲

本笔记覆盖高级数据工程师面试中最常考的 Python 语言底层知识点。

| 主题 | 出现频率 |
|------|----------|
| GIL 原理 & 多线程 vs 多进程 | 高频 |
| Generator / Iterator 协议 | 高频 |
| 装饰器 & functools.wraps | 高频 |
| Context Manager | 重要 |
| Descriptor Protocol | 重要 |
| 元类 Metaclass | 了解 |
| async/await & asyncio | 高频 |

---
## 1. GIL 原理 & 多线程 vs 多进程

### 什么是 GIL？

**GIL（Global Interpreter Lock）** 是 CPython 解释器中的一把互斥锁，确保同一时刻只有一个线程在执行 Python 字节码。

**核心原因**：CPython 的内存管理（引用计数）不是线程安全的，GIL 用最简单的方式保护它。

### GIL 对并发的影响

| 场景 | 多线程效果 | 推荐方案 |
|------|-----------|----------|
| CPU 密集型（矩阵运算、压缩） | 因 GIL 无法并行，效果差甚至更慢 | `multiprocessing` / `ProcessPoolExecutor` |
| IO 密集型（网络请求、文件读写） | 线程等待 IO 时会释放 GIL，可并发 | `threading` / `asyncio` |

### 关键释放 GIL 的时机
- 线程进入 IO 等待（`socket.recv`, `time.sleep`）
- 每执行 100 个字节码（sys.getswitchinterval，默认 5ms）
- 调用 C 扩展（如 NumPy 的内部运算）时可主动释放

In [ ]:
import threading
import multiprocessing
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# --- CPU-bound task: pure Python computation ---
def cpu_task(n: int) -> int:
    """Count from 0 to n — pure Python, GIL-bound."""
    total = 0
    for i in range(n):
        total += i
    return total

N = 10_000_000
WORKERS = 4

# Baseline: single thread
start = time.perf_counter()
for _ in range(WORKERS):
    cpu_task(N)
single_time = time.perf_counter() - start
print(f"Serial:           {single_time:.2f}s")

# Thread pool (GIL prevents true parallelism for CPU tasks)
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    list(ex.map(cpu_task, [N] * WORKERS))
thread_time = time.perf_counter() - start
print(f"ThreadPool:       {thread_time:.2f}s  (GIL bottleneck, ~same or slower)")

# Process pool (each process has its own GIL — real parallelism)
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    list(ex.map(cpu_task, [N] * WORKERS))
proc_time = time.perf_counter() - start
print(f"ProcessPool:      {proc_time:.2f}s  (real parallelism)")
print(f"Speedup (proc/serial): {single_time/proc_time:.1f}x")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# --- IO-bound task: simulated network latency ---
def io_task(delay: float) -> str:
    """Simulate IO wait — thread releases GIL during sleep."""
    time.sleep(delay)
    return f"done after {delay}s"

DELAY = 0.5
TASKS = 8

# Serial
start = time.perf_counter()
for _ in range(TASKS):
    io_task(DELAY)
print(f"Serial IO:      {time.perf_counter() - start:.2f}s")

# Thread pool: IO tasks release GIL → real concurrency
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=TASKS) as ex:
    list(ex.map(io_task, [DELAY] * TASKS))
print(f"ThreadPool IO:  {time.perf_counter() - start:.2f}s  (concurrent, ~{DELAY}s total)")

print()
print("=== Key Takeaway ===")
print("IO-bound  → threads are fine (GIL released during IO wait)")
print("CPU-bound → use processes or C-extension libraries (NumPy, etc.)")

---
## 2. Generator / Iterator 协议

### 迭代器协议

实现了以下两个方法的对象就是迭代器：
- `__iter__(self)` → 返回自身
- `__next__(self)` → 返回下一个值，耗尽时抛出 `StopIteration`

### Generator 与 Iterator 的关系

**生成器函数**（含 `yield` 的函数）调用后返回一个 **生成器对象**，生成器对象自动实现了迭代器协议。

### 为什么用 Generator？

- **惰性求值**：每次 `next()` 才计算下一个值，不需要全部放入内存
- **无限序列**：可以表示无穷数列
- **流水线处理**：多个生成器链接，数据逐条流过，内存占用极低

In [ ]:
# --- Custom Iterator class ---
class CountUp:
    """Counts from start to stop-1, implementing the iterator protocol manually."""
    def __init__(self, start: int, stop: int):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self  # an iterator's __iter__ returns itself

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

counter = CountUp(1, 6)
print("Manual iterator:", list(counter))

# Iterables vs Iterators
# A list is iterable (has __iter__) but NOT an iterator (no __next__)
my_list = [10, 20, 30]
it = iter(my_list)       # iter() calls __iter__() to get an iterator
print(next(it), next(it)) # 10 20

In [ ]:
import sys

# --- Generator function ---
def fibonacci():
    """Infinite Fibonacci sequence — only computes on demand."""
    a, b = 0, 1
    while True:
        yield a          # suspends here, sends 'a' to caller
        a, b = b, a + b  # resumes here on next next()

gen = fibonacci()
first_10 = [next(gen) for _ in range(10)]
print("First 10 Fibonacci:", first_10)

# --- Memory comparison ---
N = 1_000_000

list_mem   = sys.getsizeof(list(range(N)))          # full list in memory
gen_mem    = sys.getsizeof(x for x in range(N))     # generator object only

print(f"\nList of {N:,} ints: {list_mem:,} bytes ({list_mem/1e6:.1f} MB)")
print(f"Generator object:  {gen_mem:,} bytes")
print(f"Memory ratio:      {list_mem/gen_mem:,.0f}x")

In [ ]:
# --- yield from: delegating to sub-generators ---
def inner_gen(items):
    for item in items:
        yield item * 2

def outer_gen(groups):
    for group in groups:
        yield from inner_gen(group)   # delegates iteration to inner_gen
        # equivalent to: for v in inner_gen(group): yield v

data = [[1, 2, 3], [10, 20], [100]]
print("yield from result:", list(outer_gen(data)))

# --- Generator pipeline: process large files chunk by chunk ---
def read_lines(filename):
    """Yield lines from a file one at a time — O(1) memory."""
    with open(filename) as f:
        yield from f

def filter_nonempty(lines):
    for line in lines:
        if line.strip():
            yield line

def to_upper(lines):
    for line in lines:
        yield line.upper()

# Pipeline: read_lines → filter_nonempty → to_upper
# Each stage is a generator; data flows one line at a time
print("\nGenerator pipeline concept (chained generators):")
sample = ["hello\n", "\n", "world\n", "  \n", "python\n"]
pipeline = to_upper(filter_nonempty(iter(sample)))
for line in pipeline:
    print(repr(line))

---
## 3. 装饰器 & functools.wraps

### 装饰器本质

装饰器是一个**接收函数，返回函数**的高阶函数。`@decorator` 语法糖等价于：

```python
@decorator
def func(): ...
# 等价于
func = decorator(func)
```

### functools.wraps 的必要性

不加 `@wraps`，被装饰函数的 `__name__`、`__doc__`、`__annotations__` 等元信息会被 wrapper 覆盖，导致调试困难和文档丢失。

In [ ]:
import functools
import time

# --- Basic decorator with functools.wraps ---
def timer(func):
    """Measure and print the execution time of a function."""
    @functools.wraps(func)   # preserves __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n: int) -> int:
    """Sum of 0..n using a loop."""
    return sum(range(n))

result = slow_sum(5_000_000)
print(f"Result: {result}")
print(f"Function name preserved: {slow_sum.__name__}")  # 'slow_sum', not 'wrapper'
print(f"Docstring preserved:     {slow_sum.__doc__}")

In [ ]:
# --- Decorator with arguments (factory pattern) ---
def retry(max_attempts: int = 3, exceptions: tuple = (Exception,)):
    """Retry a function up to max_attempts times on specified exceptions."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    print(f"[retry] Attempt {attempt}/{max_attempts} failed: {e}")
            raise last_exc
        return wrapper
    return decorator

import random

@retry(max_attempts=4, exceptions=(ValueError,))
def flaky_api_call(success_rate: float = 0.5) -> str:
    """Simulates an unreliable API."""
    if random.random() > success_rate:
        raise ValueError("API timeout")
    return "API response OK"

random.seed(42)
print(flaky_api_call(success_rate=0.4))

In [ ]:
# --- Stacked decorators & class-based decorator ---
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[log] Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

# Stacking order: bottom decorator wraps first
# @log_call applied first, then @timer wraps the result
@timer
@log_call
def compute(x: int) -> int:
    return x ** 2

compute(7)

print()

# --- Class-based decorator (useful for stateful decorators) ---
class CallCounter:
    """Decorator that counts how many times a function has been called."""
    def __init__(self, func):
        functools.update_wrapper(self, func)  # same as @wraps for classes
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"[count] {self.func.__name__} called {self.count} time(s)")
        return self.func(*args, **kwargs)

@CallCounter
def greet(name: str) -> str:
    return f"Hello, {name}!"

print(greet("Alice"))
print(greet("Bob"))
print(f"Total calls: {greet.count}")

---
## 4. Context Manager — `__enter__` / `__exit__`

### 核心协议

实现了 `__enter__` 和 `__exit__` 的对象可以用于 `with` 语句：

```
with manager as value:
    body
```

等价于：
```python
value = manager.__enter__()
try:
    body
except Exception as e:
    if not manager.__exit__(type(e), e, e.__traceback__):
        raise
else:
    manager.__exit__(None, None, None)
```

`__exit__` 返回 `True` 表示**吞掉异常**，返回 `False`/`None` 表示继续向上抛出。

### 两种实现方式
1. **类方式**：适合复杂状态管理
2. **`@contextmanager`**：用 `yield` 实现，代码更简洁

In [ ]:
import time
import contextlib

# --- Class-based context manager ---
class Timer:
    """Context manager that measures elapsed time."""
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # value bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"[Timer] Elapsed: {self.elapsed:.4f}s")
        return False  # do NOT suppress exceptions

with Timer() as t:
    total = sum(range(2_000_000))
print(f"Sum: {total}, measured: {t.elapsed:.4f}s")

print()

# --- Suppressing specific exceptions ---
class SuppressErrors:
    """Suppress only specified exception types."""
    def __init__(self, *exc_types):
        self.exc_types = exc_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        suppressed = exc_type is not None and issubclass(exc_type, self.exc_types)
        if suppressed:
            print(f"[SuppressErrors] Suppressed: {exc_val}")
        return suppressed  # True → suppress, False → propagate

with SuppressErrors(ZeroDivisionError, ValueError):
    print(1 / 0)  # This error is suppressed

print("Execution continues after suppressed error")

In [ ]:
from contextlib import contextmanager
import tempfile
import os

# --- @contextmanager: generator-based context manager ---
@contextmanager
def managed_temp_file(suffix=".tmp"):
    """Create a temp file, yield its path, then clean up automatically."""
    fd, path = tempfile.mkstemp(suffix=suffix)
    os.close(fd)
    try:
        print(f"[ctx] Created temp file: {path}")
        yield path          # everything before yield = __enter__
    finally:
        os.unlink(path)     # everything in finally = __exit__
        print(f"[ctx] Deleted temp file: {path}")

with managed_temp_file(".csv") as path:
    with open(path, "w") as f:
        f.write("id,value\n1,100\n")
    print(f"[ctx] File exists during context: {os.path.exists(path)}")

print(f"File exists after context: {os.path.exists(path)}")

print()

# --- Simulated DB connection pool ---
@contextmanager
def db_connection(dsn: str):
    """Simulate acquiring and releasing a DB connection."""
    print(f"[db] Connecting to {dsn}")
    conn = {"dsn": dsn, "open": True}  # placeholder
    try:
        yield conn
        print("[db] Committing transaction")
    except Exception as e:
        print(f"[db] Rolling back: {e}")
        raise
    finally:
        conn["open"] = False
        print("[db] Connection closed")

with db_connection("postgresql://localhost/mydb") as conn:
    print(f"[db] Running query on {conn['dsn']}")

---
## 5. Descriptor Protocol — `__get__` / `__set__` / `__delete__`

### 什么是 Descriptor？

实现了以下任意方法的类实例，当被用作**类属性**时，就是一个描述符：

| 方法 | 触发时机 |
|------|----------|
| `__get__(self, obj, objtype)` | 读取属性 `obj.attr` |
| `__set__(self, obj, value)` | 设置属性 `obj.attr = val` |
| `__delete__(self, obj)` | 删除属性 `del obj.attr` |

- 同时有 `__get__` 和 `__set__` → **数据描述符**（优先级高于实例 `__dict__`）
- 只有 `__get__` → **非数据描述符**

### 典型应用
- Python 内置的 `property`、`classmethod`、`staticmethod` 都是描述符
- ORM 字段验证（SQLAlchemy Column、Django Model Field）
- 类型检查属性

In [ ]:
# --- Type-validated descriptor ---
class TypedField:
    """Descriptor that enforces a type constraint on an attribute."""

    def __init__(self, name: str, expected_type: type):
        self.name = name            # storage key in instance __dict__
        self.expected_type = expected_type

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # accessed on class, not instance
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name!r} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        obj.__dict__[self.name] = value

    def __delete__(self, obj):
        obj.__dict__.pop(self.name, None)


class DataPipeline:
    """A class using descriptors for type-safe attributes."""
    name      = TypedField("name", str)
    batch_size = TypedField("batch_size", int)
    threshold  = TypedField("threshold", float)

    def __init__(self, name: str, batch_size: int, threshold: float):
        self.name = name
        self.batch_size = batch_size
        self.threshold = threshold

# Valid usage
pipeline = DataPipeline("etl_job", 1000, 0.95)
print(f"Pipeline: {pipeline.name}, batch={pipeline.batch_size}, threshold={pipeline.threshold}")

# Type violation
try:
    pipeline.batch_size = "large"  # should raise TypeError
except TypeError as e:
    print(f"TypeError caught: {e}")

# Accessing descriptor on class returns the descriptor itself
print(f"\nClass-level access: {DataPipeline.batch_size}")

---
## 6. 元类 Metaclass

### 元类是什么？

在 Python 中，**类本身也是对象**，而创建类的「类」就是元类。

```
type(42)    → int      (int 是 42 的类)
type(int)   → type     (type 是 int 的元类)
type(type)  → type     (type 是自身的元类)
```

### `__new__` vs `__init__`

| 方法 | 用途 | 返回值 |
|------|------|--------|
| `__new__(cls, ...)` | 创建并返回新实例（或新类） | 必须返回实例 |
| `__init__(self, ...)` | 初始化已存在的实例 | 必须返回 None |

### 典型用例
- 自动注册子类（插件系统、序列化框架）
- 强制类必须实现某些方法（比 ABC 更灵活）
- ORM 模型定义（Django、SQLAlchemy）

In [ ]:
# --- Metaclass for auto-registering subclasses (plugin pattern) ---
class PluginMeta(type):
    """Metaclass that automatically registers all subclasses in a registry."""
    registry: dict[str, type] = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        # Don't register the base class itself
        if bases:  # bases is empty for the root class
            mcs.registry[name] = cls
            print(f"[PluginMeta] Registered: {name}")
        return cls


class BaseTransformer(metaclass=PluginMeta):
    """Base class for all data transformers."""
    def transform(self, data):
        raise NotImplementedError


class UpperCaseTransformer(BaseTransformer):
    def transform(self, data):
        return data.upper()


class StripTransformer(BaseTransformer):
    def transform(self, data):
        return data.strip()


print(f"\nAll registered transformers: {list(PluginMeta.registry.keys())}")

# Instantiate by name (like a factory pattern)
def get_transformer(name: str) -> BaseTransformer:
    cls = PluginMeta.registry.get(name)
    if cls is None:
        raise KeyError(f"Unknown transformer: {name}")
    return cls()

t = get_transformer("UpperCaseTransformer")
print(f"Transform result: {t.transform('  hello world  ')}")

# --- __new__ vs __init__ ---
class Singleton:
    """Classic Singleton using __new__."""
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance  # always returns the SAME object

a = Singleton()
b = Singleton()
print(f"\nSingleton: a is b → {a is b}")

---
## 7. async/await & asyncio 事件循环

### 核心概念

| 术语 | 含义 |
|------|------|
| **协程 (coroutine)** | `async def` 定义的函数，调用后返回协程对象，必须被 await |
| **事件循环 (event loop)** | asyncio 的核心调度器，单线程轮询 IO 事件并调度协程 |
| **Task** | 封装协程，由事件循环并发执行的调度单元 |
| **await** | 挂起当前协程，将控制权交还事件循环，等待结果 |

### asyncio vs threading（IO 密集型）

- **threading**：每个并发任务一个线程，有创建开销，受 GIL 影响，适合 10~100 并发
- **asyncio**：单线程协作式，切换开销极小，适合 10,000+ 并发 IO 任务
- **关键区别**：asyncio 是**协作式**多任务（需要 await 主动让出），threading 是**抢占式**

In [ ]:
import asyncio
import time

# --- Basic coroutine and gather ---
async def fetch_data(source: str, delay: float) -> dict:
    """Simulates an async IO operation (e.g., HTTP request)."""
    print(f"  [{source}] Starting fetch...")
    await asyncio.sleep(delay)   # non-blocking sleep — releases event loop
    print(f"  [{source}] Done after {delay}s")
    return {"source": source, "rows": 42}

async def main_gather():
    """Run multiple coroutines concurrently with asyncio.gather."""
    start = time.perf_counter()

    # gather runs all coroutines concurrently (not sequentially)
    results = await asyncio.gather(
        fetch_data("database",  0.3),
        fetch_data("api_v1",    0.5),
        fetch_data("api_v2",    0.4),
        fetch_data("s3_bucket", 0.2),
    )

    elapsed = time.perf_counter() - start
    print(f"\nAll done in {elapsed:.2f}s (max delay was 0.5s)")
    print(f"Results: {results}")

# In Jupyter, the event loop is already running — use await directly
await main_gather()

In [ ]:
import asyncio

# --- Async generator ---
async def async_batch_reader(total: int, batch_size: int):
    """Async generator that yields batches of records."""
    for offset in range(0, total, batch_size):
        await asyncio.sleep(0.01)   # simulate async DB query
        batch = list(range(offset, min(offset + batch_size, total)))
        yield batch

async def process_all_batches():
    total_processed = 0
    async for batch in async_batch_reader(total=25, batch_size=10):
        total_processed += len(batch)
        print(f"Processed batch: {batch[:3]}... ({len(batch)} records)")
    print(f"Total processed: {total_processed} records")

await process_all_batches()

print()

# --- asyncio.TaskGroup (Python 3.11+) for structured concurrency ---
async def risky_task(name: str, should_fail: bool = False):
    await asyncio.sleep(0.1)
    if should_fail:
        raise RuntimeError(f"{name} failed!")
    return f"{name} succeeded"

async def run_task_group():
    try:
        async with asyncio.TaskGroup() as tg:
            t1 = tg.create_task(risky_task("task_A"))
            t2 = tg.create_task(risky_task("task_B"))
            # All tasks complete or all are cancelled on first failure
        print(f"task_A: {t1.result()}")
        print(f"task_B: {t2.result()}")
    except* RuntimeError as eg:
        print(f"TaskGroup errors: {eg.exceptions}")

await run_task_group()

In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

# --- Run blocking code in thread pool from async context ---
def blocking_io(duration: float) -> str:
    """Simulates a blocking library call (e.g., old SDK without async support)."""
    time.sleep(duration)
    return f"blocking done after {duration}s"

async def async_with_executor():
    loop = asyncio.get_running_loop()
    # run_in_executor offloads blocking calls to a thread pool
    # This keeps the event loop unblocked
    with ThreadPoolExecutor(max_workers=3) as pool:
        results = await asyncio.gather(
            loop.run_in_executor(pool, blocking_io, 0.3),
            loop.run_in_executor(pool, blocking_io, 0.4),
            loop.run_in_executor(pool, blocking_io, 0.2),
        )
    return results

start = time.perf_counter()
results = await async_with_executor()
print(f"run_in_executor results: {results}")
print(f"Total time: {time.perf_counter() - start:.2f}s (concurrent, ~0.4s max)")

---
## 复习要点

### GIL
- GIL 只存在于 CPython，保护引用计数，同时只有一个线程执行字节码
- IO 密集 → 线程/asyncio；CPU 密集 → multiprocessing / C 扩展
- `concurrent.futures` 提供统一 API：`ThreadPoolExecutor` / `ProcessPoolExecutor`

### Generator
- 迭代器协议：`__iter__` + `__next__`，耗尽抛 `StopIteration`
- `yield` 暂停函数，`yield from` 委托子生成器
- 生成器 vs 列表：内存 O(1) vs O(n)，适合流式处理大数据

### 装饰器
- 本质是高阶函数：`decorator(func)` → 新函数
- `@functools.wraps(func)` 保留元信息（必须加）
- 带参数的装饰器 = 三层嵌套；类装饰器用 `__call__` 实现

### Context Manager
- `__enter__` 返回资源，`__exit__` 负责清理（即使异常也会执行）
- `__exit__` 返回 True → 吞掉异常
- `@contextmanager` 用 `yield` 分隔 enter/exit 逻辑

### Descriptor
- 数据描述符（有 `__set__`）优先级 > 实例 `__dict__` > 非数据描述符
- `property`、`classmethod`、`staticmethod` 都是描述符
- 用于 ORM 字段、类型验证、缓存属性

### Metaclass
- `type` 是所有类的元类，可自定义以拦截类创建
- `__new__` 创建对象，`__init__` 初始化对象
- 常用于插件注册、ORM、强制接口约定

### asyncio
- 单线程事件循环，协作式调度，适合高并发 IO
- `asyncio.gather` 并发执行，`TaskGroup` 结构化并发（3.11+）
- 阻塞调用用 `run_in_executor` 放入线程池，避免阻塞事件循环
- `async for` 消费异步生成器

---
## 练习

以下练习覆盖本章所有主题，建议先独立完成再参考参考答案。

### 练习 1 — GIL & 并发

下面的代码想并行计算 4 个大列表的元素之和，但作者直接用了 `ThreadPoolExecutor`。

**任务**：
1. 解释为什么这对 CPU 密集型任务效果不好
2. 将其改为 `ProcessPoolExecutor`，并用 `time.perf_counter` 比较两者耗时
3. 思考：如果把 `cpu_sum` 改成 `numpy.sum(arr)`，线程版会变快吗？为什么？

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def cpu_sum(n):
    return sum(range(n))

# TODO: 1. Run with ThreadPoolExecutor and measure time
# TODO: 2. Run with ProcessPoolExecutor and measure time
# TODO: 3. Compare and explain the difference

N_LIST = [8_000_000] * 4

# Your code here


### 练习 2 — Generator

**任务**：实现一个生成器函数 `chunked_reader(data, chunk_size)`，
它接收一个可迭代对象和块大小，每次 `yield` 一个列表块（最后一块可能不满）。

例如：`list(chunked_reader(range(10), 3))` → `[[0,1,2], [3,4,5], [6,7,8], [9]]`

然后用它模拟分批从数据库读取 100 万行数据，计算每批的总和，最终合计。

In [ ]:
# TODO: Implement chunked_reader generator
def chunked_reader(data, chunk_size: int):
    pass

# Test
print(list(chunked_reader(range(10), 3)))  # Expected: [[0,1,2], [3,4,5], [6,7,8], [9]]

# TODO: Use chunked_reader to process 1M rows in batches of 10,000
# Compute grand total without loading all data at once


### 练习 3 — 装饰器

**任务**：实现一个带参数的装饰器 `@cache(max_size=N)`，功能类似 `functools.lru_cache`：
- 缓存最近 N 次调用的结果（按参数为 key）
- 缓存满时，淘汰最久未使用的条目（LRU 策略）
- 用 `functools.wraps` 保留原函数元信息
- 添加 `.cache_info()` 方法返回命中率统计

用它装饰一个递归 Fibonacci 函数，验证缓存效果。

In [ ]:
# Hint: use collections.OrderedDict for LRU eviction
from collections import OrderedDict
import functools

def cache(max_size: int = 128):
    # TODO: Implement LRU cache decorator
    pass

@cache(max_size=50)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

# Test
print(fib(30))
# print(fib.cache_info())  # Uncomment after implementing


### 练习 4 — Context Manager

**任务**：用 `@contextmanager` 实现一个 `atomic_write(filepath)` 上下文管理器：
- 写入时先写到同目录下的临时文件
- `with` 块正常退出后，将临时文件**原子性地**重命名为目标文件（`os.replace`）
- 如果 `with` 块内抛出异常，删除临时文件，不影响原文件

这是生产数据管道中常见的**防止文件写到一半崩溃**的模式。

In [ ]:
from contextlib import contextmanager
import os
import tempfile

@contextmanager
def atomic_write(filepath: str, mode: str = "w"):
    # TODO: Implement atomic write context manager
    pass

# Test: successful write
# with atomic_write("/tmp/test_output.csv") as f:
#     f.write("id,value\n1,100\n")

# Test: failed write (original file should be untouched)
# with atomic_write("/tmp/test_output.csv") as f:
#     f.write("partial data...")
#     raise RuntimeError("Simulated crash")


### 练习 5 — Descriptor

**任务**：实现一个 `RangeField` 描述符，要求：
- 只接受 `int` 或 `float` 类型
- 值必须在 `[min_val, max_val]` 范围内
- 超出范围时抛出 `ValueError` 并说明具体范围

用它定义一个 `ModelConfig` 类：
- `learning_rate`: float, 范围 `[1e-6, 1.0]`
- `num_epochs`: int, 范围 `[1, 1000]`
- `dropout`: float, 范围 `[0.0, 0.9]`

In [ ]:
class RangeField:
    # TODO: Implement range-validated descriptor
    pass

class ModelConfig:
    learning_rate = RangeField("learning_rate", 1e-6, 1.0)
    num_epochs    = RangeField("num_epochs", 1, 1000)
    dropout       = RangeField("dropout", 0.0, 0.9)

    def __init__(self, lr, epochs, dropout):
        self.learning_rate = lr
        self.num_epochs = epochs
        self.dropout = dropout

# Test valid config
# cfg = ModelConfig(0.001, 100, 0.3)

# Test invalid config
# cfg = ModelConfig(5.0, 100, 0.3)  # learning_rate out of range


### 练习 6 — Metaclass

**任务**：创建一个元类 `InterfaceMeta`，强制所有子类必须实现指定的抽象方法列表（存储在类属性 `REQUIRED_METHODS` 中）。

如果子类缺少任何必要方法，在类**创建时**（而非实例化时）就抛出 `TypeError`。

```python
class DataSource(metaclass=InterfaceMeta):
    REQUIRED_METHODS = ["connect", "read", "close"]

class BadSource(DataSource):
    def connect(self): ...
    # Missing 'read' and 'close' — should raise TypeError at class definition
```

In [ ]:
class InterfaceMeta(type):
    # TODO: Implement interface enforcement metaclass
    pass

class DataSource(metaclass=InterfaceMeta):
    REQUIRED_METHODS = ["connect", "read", "close"]

# This should raise TypeError immediately:
# class BadSource(DataSource):
#     def connect(self): pass

# This should work fine:
# class GoodSource(DataSource):
#     def connect(self): print("connecting")
#     def read(self):    return []
#     def close(self):   print("closing")


### 练习 7 — asyncio

**任务**：实现一个异步批量 HTTP 请求模拟器：

1. 定义 `async def fetch(url, session)` 模拟 HTTP GET（用 `asyncio.sleep` 模拟延迟，随机 0.1~0.5 秒）
2. 使用 **信号量（asyncio.Semaphore）** 限制同时进行的请求数不超过 5
3. 对 20 个 URL 并发请求，收集所有结果
4. 用 `asyncio.gather(..., return_exceptions=True)` 处理部分失败的情况（随机让 20% 的请求失败）
5. 最终打印：成功数、失败数、总耗时

In [ ]:
import asyncio
import random
import time

# TODO: Implement rate-limited async HTTP fetcher

async def fetch(url: str, semaphore: asyncio.Semaphore) -> dict:
    """Fetch a URL with concurrency limited by semaphore."""
    # TODO: acquire semaphore, simulate request, maybe raise on failure
    pass

async def batch_fetch(urls: list[str], max_concurrent: int = 5):
    """Fetch all URLs with limited concurrency."""
    # TODO: create semaphore, gather all tasks, count successes/failures
    pass

# Test
urls = [f"https://api.example.com/data/{i}" for i in range(20)]
random.seed(42)
# await batch_fetch(urls, max_concurrent=5)
